## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries for Llama 3.1 support.

## Step 1: Configuration

All paths and variables for easy configuration.

In [1]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================

# Base model configuration
BASE_LLM = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MODEL_NAME_BASE = "stoic_llama31_8b_instruct_unsloth_4bit_seneca"

# Input data configuration
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/seneca_dataset"

# Output directory structure - all under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"       # LoRA weights during training
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/model_4bit"    # 4-bit merged model
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"            # GGUF for Ollama

# Training configuration
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
TARGET_EPOCHS = 1

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# GGUF conversion configuration
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"
QUANTIZATION_TYPE = "q4_k"  # Options: None, "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"

# vLLM deployment
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

print("✓ Configuration loaded")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")

✓ Configuration loaded
  Base model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
  Model name: stoic_llama31_8b_instruct_unsloth_4bit_seneca
  Input data: /home/spark/projects/augmentoolkit/outputs/seneca_dataset
  Output base: ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca


In [2]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/spark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/spark/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth: 2026.1.4
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Seneca dataset (first-person Stoic responses from Letters and Essays).

In [3]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)
dataset = dataset.shuffle(seed=42)

print(f"\n✓ Total: {len(dataset)} examples from ALL Augmentoolkit output")
print(f"✓ Columns: {dataset.column_names}")

import json
print("\n--- Sample ---")
print(json.dumps(dataset[0], indent=2)[:500])

📚 LOADING ALL AUGMENTOOLKIT DATA
Found 50 subdirectories


Generating train split: 101 examples [00:00, 9880.69 examples/s]


  Loaded 101 from factual_sft_stoics_seneca_followup_0/simplified_data_rag.jsonl


Generating train split: 101 examples [00:00, 39591.09 examples/s]


  Loaded 101 from factual_sft_stoics_seneca_followup_0/plain_qa_list.jsonl


...

  Loaded 2508 from representation_variation_stoics_seneca/final_output.jsonl


Generating train split: 105 examples [00:00, 20628.69 examples/s]


  Loaded 105 from sft_run/axolotl_rag_conversations_stoics_seneca.jsonl


Generating train split: 3 examples [00:00, 433.19 examples/s]


  Loaded 3 from sft_run/pretraining_subset_478099.jsonl

✓ Total: 30756 examples from ALL Augmentoolkit output
✓ Columns: ['conversations', 'text', 'metadata', 'segments', 'question', 'source_text', 'source_metadata', 'related_chunks']

--- Sample ---
{
  "conversations": [
    {
      "from": "system",
      "value": "does not matter, RAG in this pipeline is deprecated, there's a separate pipeline for that now"
    },
    {
      "from": "human",
      "value": "What are the key characteristics of a person who has achieved a state of inner strength and resilience, and how do they navigate the challenges of life, including pain, death, and the influence of others?"
    },
    {
      "from": "gpt",
      "value": "I have learned that a person


## 4. Load Model & Tokenizer with Unsloth

Load Llama 3.1 8B **Instruct** model for system prompt flexibility - allows persona-switching between different Stoic philosophers.

In [4]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map="auto"     # Automatic device placement
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: 4-bit (QLoRA)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✓ Model loaded: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
✓ Precision: 4-bit (QLoRA)
✓ Tokenizer configured
✓ Max sequence length: 2048


In [5]:
# Format dataset for Llama 3.1 chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"✓ Dataset formatted: {len(dataset)} examples")
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])

Filter: 100%|██████████| 30756/30756 [00:00<00:00, 759381.98 examples/s]

✓ Dataset formatted: 18026 examples

--- Sample formatted text (first 500 chars) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

does not matter, RAG in this pipeline is deprecated, there's a separate pipeline for that now<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the key characteristics of a person who has achieved a state of inner strength and resilience, and how do they navigate the challenges of life, including pain, death, and the influence of others?<|eot_id|><|start_header_id


## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning with attention and MLP projection layers.

In [6]:
from peft import LoraConfig

# Conservative LoRA: lower rank + attention-only for gentle adaptation
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA adapters added (r=16, targets=['q_proj', 'v_proj'])
✓ Trainable parameters: 6,815,744


## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Stoic examples from Meditations
- 1 epoch with low learning rate to gently teach persona
- This preserves base model capabilities while adding authentic voice

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = steps_per_epoch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for 4-bit QLoRA training")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")
print(f"✓ Output directory: {OUTPUT_DIR_ADAPTERS}")

Unsloth: Tokenizing ["text"] (num_proc=24): 100%|██████████| 18026/18026 [00:04<00:00, 3921.44 examples/s]

✓ Trainer configured for 4-bit QLoRA training
✓ Dataset size: 18026 conversations
✓ Effective batch size: 8 (batch=2 × grad_accum=4)
✓ Steps per epoch: 2253
✓ Total epochs: 1
✓ Total steps: 2253
✓ Warmup steps: 225
✓ Save every: 2253 steps (every epoch)
✓ Output directory: ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca/train


In [8]:
# Start training
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,026 | Num Epochs = 1 | Total steps = 2,253
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,815,744 of 8,037,076,992 (0.08% trained)


Step,Training Loss
10,2.272700
20,2.359900
30,2.382400
40,2.319900
50,2.292900
60,2.274500
70,2.254800
80,1.997900
90,2.177300
100,2.162700


TrainOutput(global_step=2253, training_loss=1.57070556112358, metrics={'train_runtime': 8212.2558, 'train_samples_per_second': 2.195, 'train_steps_per_second': 0.274, 'total_flos': 5.381007904266486e+17, 'train_loss': 1.57070556112358, 'epoch': 0.999889049151226})

## 7. Save Model & Inference

Save the fine-tuned model and test inference with a Stoic question.

In [9]:
from pathlib import Path
import json

# FIRST: Save LoRA adapters while they still exist (BEFORE merging!)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"💾 Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"✅ LoRA adapters saved (can be loaded on any Llama 3.1 8B model)")

# THEN: Save merged 4-bit model
Path(OUTPUT_DIR_MERGED).mkdir(parents=True, exist_ok=True)
print(f"\n💾 Saving merged 4-bit model to {OUTPUT_DIR_MERGED}...")

model.save_pretrained_merged(
    OUTPUT_DIR_MERGED,
    tokenizer,
    save_method="merged_4bit_forced"  # 4-bit safetensors output
)

# Fix tokenizer for vLLM compatibility
LLAMA31_CHAT_TEMPLATE = """{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"""

tokenizer_config_path = Path(OUTPUT_DIR_MERGED) / "tokenizer_config.json"
if tokenizer_config_path.exists():
    with open(tokenizer_config_path, "r") as f:
        tokenizer_config = json.load(f)
    
    if "chat_template" not in tokenizer_config or not tokenizer_config["chat_template"]:
        tokenizer_config["chat_template"] = LLAMA31_CHAT_TEMPLATE
        with open(tokenizer_config_path, "w") as f:
            json.dump(tokenizer_config, f, indent=2)
        print("✓ Added Llama 3.1 chat template")

print("✓ tokenizer_config.json updated")
print(f"   Format: safetensors (bitsandbytes 4-bit)")
print(f"   Tokenizer: Fixed for vLLM compatibility")
print(f"\n✅ 4-bit merged model saved to {OUTPUT_DIR_MERGED}")

💾 Saving LoRA adapters to ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca/lora_adapters...
✅ LoRA adapters saved (can be loaded on any Llama 3.1 8B model)

💾 Saving merged 4-bit model to ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca/model_4bit...
Unsloth: Merging LoRA weights into 4bit model...


/home/spark/.venv/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Unsloth: Merging finished.
Unsloth: Found skipped modules: ['lm_head']. Updating config.
Unsloth: Saving merged 4bit model to ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca/model_4bit...
Unsloth: Merged 4bit model saved.
Unsloth: Merged 4bit model process completed.
✓ tokenizer_config.json updated
   Format: safetensors (bitsandbytes 4-bit)
   Tokenizer: Fixed for vLLM compatibility

✅ 4-bit merged model saved to ./output/stoic_llama31_8b_instruct_unsloth_4bit_seneca/model_4bit


In [10]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference with a Stoic question
test_prompt = "I am troubled by the loss of my possessions. How should I think about this?"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7, 
    top_p=0.9,
    repetition_penalty=1.1
)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\n=== RAW FULL OUTPUT (with tags) ===")
print(response)
print("\n=== PARSED OUTPUT ===")
print(f"User: {test_prompt}")

# Extract just the assistant response (after the last header)
# Llama 3.1 format: <|start_header_id|>assistant<|end_header_id|>\n\nRESPONSE<|eot_id|>
if "<|start_header_id|>assistant<|end_header_id|>" in response:
    assistant_response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    assistant_response = assistant_response.replace("<|eot_id|>", "").strip()
    print(f"\nAssistant: {assistant_response}")
else:
    # Fallback for cleaner display
    print(f"\nAssistant: {tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== RAW FULL OUTPUT (with tags) ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

I am troubled by the loss of my possessions. How should I think about this?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Losing one's possessions can be a difficult and emotional experience, especially if they hold sentimental value or were essential to your daily life.

Here are some suggestions on how to approach this situation:

1. **Acknowledge your feelings**: It's normal to feel sad, angry, or frustrated when losing possessions. Recognize these emotions and give yourself permission to process them.
2. **Focus on what you have left**: Rather than dwelling on what you've lost, try to focus on the things you still possess. Make a mental or physical inventory of the items that remain with you.
3. **Practice gratitude**: Take time to appreciate the items you do hav

## Notes

### Configuration
All paths and variables are configured in Step 1 for easy modification:
- `BASE_LLM`: Base model to fine-tune
- `MODEL_NAME_BASE`: Name used for all output folders
- `INPUT_DATA_PATH`: Source data from Augmentoolkit
- Output structure: `./output/{MODEL_NAME_BASE}/{adapters|merged|gguf}`

### Dataset Quality
This notebook uses Augmentoolkit-generated data from Seneca's works (Letters to Lucilius and Essays). The pipeline enforced first-person responses through configuration:
- `shared_instruction`: "You ARE a Stoic philosopher - not explaining Stoicism, but LIVING it."
- Source texts: Letters to Lucilius and Moral Essays (wisdom through practical advice)
- All prompts rewritten to enforce "I am a Stoic philosopher..." voice

### Next Steps
- For other philosophers: Run Augmentoolkit with Epictetus (Enchiridion/Discourses) or Marcus Aurelius (Meditations)
- Merge datasets: Combine multiple Stoic philosophers for broader knowledge
- Evaluate first-person quality: Test if model says "I practice..." vs "Stoics believe..."